# Homework 2: Data Manipulation

In this homework, you'll practice filtering, transforming, aggregating, and joining data with Polars.

## Dataset: Retail Sales

You'll work with a retail sales dataset containing transactions from multiple stores across different regions.

## Instructions

- Complete each exercise in the provided code cells
- Use the Polars expression API
- Points are indicated for each exercise

In [2]:
import polars as pl

## Exercise 1: Load and Inspect Data (10 points)

1. Load the `retail_sales.csv` file
2. Display the first 5 rows
3. Show the schema and check for null values

In [3]:
# Load data
sales = pl.read_csv("https://raw.githubusercontent.com/Navenkumar-Balasubramaniam/00-General/refs/heads/main/18%20Polars/exercise%202/retail_sales.csv")

In [4]:
# Display first 5 rows
sales.head(5)

sale_id,store_id,region,product_name,category,quantity,unit_price,sale_date,customer_age_group
i64,str,str,str,str,i64,f64,str,str
1,"""Store_028""","""East""","""Yogurt""","""Dairy""",2,2.49,"""2024-09-09""","""46-55"""
2,"""Store_002""","""East""","""Apple""","""Produce""",2,0.79,"""2024-10-25""","""26-35"""
3,"""Store_048""","""East""","""Pork""","""Meat""",1,7.99,"""2024-06-23""","""46-55"""
4,"""Store_028""","""West""","""Croissant""","""Bakery""",2,1.99,"""2024-06-12""","""36-45"""
5,"""Store_013""","""North""","""Milk""","""Dairy""",2,3.99,"""2023-10-05""","""18-25"""


In [5]:
# Show schema and null counts
sales.schema

Schema([('sale_id', Int64),
        ('store_id', String),
        ('region', String),
        ('product_name', String),
        ('category', String),
        ('quantity', Int64),
        ('unit_price', Float64),
        ('sale_date', String),
        ('customer_age_group', String)])

In [6]:
sales.null_count()

sale_id,store_id,region,product_name,category,quantity,unit_price,sale_date,customer_age_group
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,197,0,536


## Exercise 2: Filtering (15 points)

1. Find all sales from the "North" region with quantity >= 10
2. Find all sales of Dairy or Meat products (use `is_in()`)
3. Find sales where `unit_price` is not null AND category is "Produce"

In [8]:
# 2.1 North region, quantity >= 10
north_large = sales.filter(
    (pl.col("region") == "North") &
     (pl.col("quantity") >= 10)
     )

In [11]:
# 2.2 Dairy or Meat products
dairy_meat = sales.filter(
    pl.col("category").is_in(["Dairy", "Meat"])
    )

In [13]:
# 2.3 Non-null price AND Produce category
produce_valid = sales.filter(
    (pl.col("unit_price").is_not_null()) &
    (pl.col("category") == "Produce")
    )

## Exercise 3: Creating Columns (15 points)

Create a new DataFrame with the following additional columns:

1. `total_revenue`: quantity * unit_price (handle nulls by using 0 for null prices)
2. `quantity_category`: "High" if quantity >= 15, "Medium" if >= 5, "Low" otherwise
3. `product_name_clean`: product_name in lowercase with leading/trailing spaces removed

In [14]:
# Your code here
sales_enhanced = sales.with_columns(
    total_revenue = pl.col("quantity") * pl.col("unit_price").fill_null(0),
    quantity_category = pl.when(pl.col("quantity") >= 15).then(pl.lit("High"))
                                      .when(pl.col("quantity") >= 5).then(pl.lit("Medium"))
                                      .otherwise(pl.lit("Low")),
    product_name_clean = pl.col("product_name").str.to_lowercase().str.strip_chars()
)

In [15]:
sales_enhanced.head(5)

sale_id,store_id,region,product_name,category,quantity,unit_price,sale_date,customer_age_group,total_revenue,quantity_category,product_name_clean
i64,str,str,str,str,i64,f64,str,str,f64,str,str
1,"""Store_028""","""East""","""Yogurt""","""Dairy""",2,2.49,"""2024-09-09""","""46-55""",4.98,"""Low""","""yogurt"""
2,"""Store_002""","""East""","""Apple""","""Produce""",2,0.79,"""2024-10-25""","""26-35""",1.58,"""Low""","""apple"""
3,"""Store_048""","""East""","""Pork""","""Meat""",1,7.99,"""2024-06-23""","""46-55""",7.99,"""Low""","""pork"""
4,"""Store_028""","""West""","""Croissant""","""Bakery""",2,1.99,"""2024-06-12""","""36-45""",3.98,"""Low""","""croissant"""
5,"""Store_013""","""North""","""Milk""","""Dairy""",2,3.99,"""2023-10-05""","""18-25""",7.98,"""Low""","""milk"""


## Exercise 4: Groupby Aggregations (20 points)

1. Calculate total revenue and average quantity by `region`
2. Find the top 5 products by total quantity sold
3. Calculate the number of sales and unique stores per category

In [16]:
# 4.1 Revenue and avg quantity by region
region_stats = sales_enhanced.group_by("region").agg(pl.col("total_revenue").sum().alias("Total_Revenue"), pl.col("quantity").mean().alias("Average_Quantity"))
region_stats

region,Total_Revenue,Average_Quantity
str,f64,f64
"""West""",101892.12,10.420566
"""South""",102565.45,10.619766
"""East""",99584.79,10.500503
"""North""",100551.9,10.412241
"""Central""",103560.38,10.658153


In [20]:
# 4.2 Top 5 products by quantity
top_products = sales_enhanced.group_by("product_name_clean").agg(pl.col("quantity").sum().alias("Total_Quantity")).sort(by=pl.col("Total_Quantity"),descending=True).limit(5)
top_products

product_name_clean,Total_Quantity
str,i64
"""pork""",5698
"""bread""",5646
"""juice""",5528
"""pasta""",5299
"""milk""",5169


In [22]:
# 4.3 Sales count and unique stores per category
category_stats = sales_enhanced.group_by("category").agg(pl.col("store_id").n_unique().alias("Unique_Stores"), pl.col("sale_id").count().alias("Sales_Count"))
category_stats

category,Unique_Stores,Sales_Count
str,u32,u32
"""Beverages""",50,1450
"""Dairy""",50,1431
"""Seafood""",50,1419
"""Pantry""",50,1443
"""Produce""",50,1361
"""Meat""",50,1457
"""Bakery""",50,1439


## Exercise 5: Handling Missing Data (15 points)

1. Count how many rows have null values in `unit_price` or `customer_age_group`
2. Create a cleaned DataFrame where:
   - Rows with null `unit_price` are removed
   - Null `customer_age_group` is filled with "Unknown"
3. Verify no nulls remain in these columns

In [23]:
# 5.1 Count rows with nulls
count_nulls = sales.filter(
    pl.col("unit_price").is_null() |
    pl.col("customer_age_group").is_null()
).height

print(count_nulls)

720


In [26]:
# 5.2 Create cleaned DataFrame
sales_clean = sales.filter(
    pl.col("unit_price").is_not_null()
    ).with_columns(
    customer_age_group = pl.col("customer_age_group").fill_null("Unknown")
    )

In [27]:
# 5.3 Verify no nulls
sales_clean.null_count()

sale_id,store_id,region,product_name,category,quantity,unit_price,sale_date,customer_age_group
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0


## Exercise 6: Sorting and Top-N (10 points)

1. Sort sales by region (ascending) then by quantity (descending)
2. Get the 10 highest-value transactions (by total_revenue from Exercise 3)

In [ ]:
# 6.1 Sort by region and quantity
sorted_sales = ...

In [ ]:
# 6.2 Top 10 by revenue
top_transactions = ...

## Exercise 7: Advanced Analysis (15 points)

Perform a comprehensive analysis:

1. Calculate revenue by region and category (multi-level groupby)
2. Find the best-selling product in each region
3. Calculate the percentage of total sales for each category

In [ ]:
# 7.1 Revenue by region and category
region_category = ...

In [ ]:
# 7.2 Best-selling product per region
best_per_region = ...

In [ ]:
# 7.3 Category percentage of total
category_pct = ...

## Bonus: Store Performance Report (10 bonus points)

Create a comprehensive store performance report showing:
- Store ID
- Region
- Total sales count
- Total revenue
- Average transaction value
- Number of unique products sold
- Most common category

Sort by total revenue descending and show top 10 stores.

In [ ]:
# Bonus: Store performance report
store_report = ...